In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/zodasic/test/pytorch/default/1/nguyen_tac_phan_chia_data_client.py
/kaggle/input/models/zodasic/test2/pytorch/default/1/unified_fl_config.py
/kaggle/input/datasets/zodasic/road-pro/Dataset/stats.json
/kaggle/input/datasets/zodasic/road-pro/Dataset/concept_1.parquet
/kaggle/input/datasets/zodasic/road-pro/Dataset/road_multi_label.csv
/kaggle/input/datasets/zodasic/road-pro/Dataset/train.parquet
/kaggle/input/datasets/zodasic/road-pro/Dataset/concept_3.parquet
/kaggle/input/datasets/zodasic/road-pro/Dataset/concept_2.parquet
/kaggle/input/datasets/zodasic/road-pro/Dataset/test.parquet
/kaggle/input/datasets/zodasic/road-pre/road/readme.md
/kaggle/input/datasets/zodasic/road-pre/road/data_table.csv
/kaggle/input/datasets/zodasic/road-pre/road/attacks/accelerator_attack_drive_2.log
/kaggle/input/datasets/zodasic/road-pre/road/attacks/correlated_signal_attack_1.log
/kaggle/input/datasets/zodasic/road-pre/road/attacks/reverse_light_off_attack_2_masquerade.log
/kaggle/inp

In [2]:
from __future__ import annotations

import copy
import importlib.util
import json
import logging
import math
import random
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, Dataset, Subset

# Required paths
DATA_PATH = "/kaggle/input/datasets/zodasic/road-pro/Dataset/road_multi_label.csv"
CONFIG_PATH = "/kaggle/input/models/zodasic/test2/pytorch/default/1/unified_fl_config.py"
SPLIT_SCRIPT_PATH = "/kaggle/input/models/zodasic/test/pytorch/default/1/nguyen_tac_phan_chia_data_client.py"
OUTPUT_DIR = "/kaggle/working/MNAS"
PAPER_PATH = "" # Placeholder for paper path

OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "outputs").mkdir(parents=True, exist_ok=True)
(OUT / "outputs" / "metrics").mkdir(parents=True, exist_ok=True)
(OUT / "outputs" / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "outputs" / "checkpoints").mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("mnas_impl")

## Print/Progress Utilities

Yêu cầu print/log tiến trình được hiện thực rõ ràng để dễ theo dõi khi bạn tự chạy notebook.


In [3]:
def print_section(title: str) -> None:
    print("\n" + "=" * 110)
    print(f"[SECTION] {title}")
    print("=" * 110)


def print_subsection(title: str) -> None:
    print("\n" + "-" * 90)
    print(f"[STEP] {title}")
    print("-" * 90)


def print_dataset_summary(summary: Dict[str, Any]) -> None:
    print_subsection("Dataset Summary")
    for k, v in summary.items():
        print(f"- {k}: {v}")


def print_client_split_summary(num_clients: int, stat_df: pd.DataFrame, extra: Dict[str, Any]) -> None:
    print_subsection(f"Client Split Summary ({num_clients} clients)")
    print(stat_df.head(min(10, len(stat_df))).to_string(index=False))
    for k, v in extra.items():
        print(f"- {k}: {v}")


def print_round_start(round_idx: int, total_rounds: int, num_selected_clients: int) -> None:
    now = time.strftime("%H:%M:%S")
    print(f"[{now}] [ROUND {round_idx}/{total_rounds}] start | selected_clients={num_selected_clients}")


def print_client_training_progress(round_idx: int, client_pos: int, total_clients: int, client_id: int, mode: str, loss_personal: float, loss_proxy: float) -> None:
    now = time.strftime("%H:%M:%S")
    print(
        f"[{now}]   client_progress={client_pos}/{total_clients} | round={round_idx} | "
        f"client_id={client_id} | mode={mode} | loss_personal={loss_personal:.6f} | loss_proxy={loss_proxy:.6f}"
    )


def print_round_metrics(round_idx: int, metrics: Dict[str, float], n_ptq: int = 0, n_qat: int = 0) -> None:
    # n_ptq/n_qat kept for interface compatibility; MNAS does not use PTQ/QAT dispatch.
    now = time.strftime("%H:%M:%S")
    print(
        f"[{now}] [ROUND {round_idx}] loss={metrics['loss']:.6f} | "
        f"precision={metrics['precision']:.6f} | recall={metrics['recall']:.6f} | f1={metrics['f1']:.6f}"
    )


def print_experiment_summary(num_clients: int, final_metrics: Dict[str, float], elapsed_sec: float) -> None:
    print_subsection(f"Experiment Summary ({num_clients} clients)")
    print(f"- elapsed_sec: {elapsed_sec:.2f}")
    print(f"- loss: {final_metrics['loss']:.6f}")
    print(f"- precision: {final_metrics['precision']:.6f}")
    print(f"- recall: {final_metrics['recall']:.6f}")
    print(f"- f1: {final_metrics['f1']:.6f}")


## Reproducibility + Config Integration


In [4]:
import sys
from dataclasses import dataclass, field, asdict
import importlib.util
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import torch


@dataclass
class MNASRuntimeConfig:
    seed: int
    device: str
    data_path: str
    output_dir: str

    scenarios: Tuple[int, ...]
    dirichlet_alpha: float
    min_client_samples: int
    partition_retry_attempts: int

    # approximate-equal constraints
    rebalance_tolerance_ratio: float
    required_min_coverage_ratio: float

    # data processing
    batch_size: int
    num_workers: int
    pin_memory: bool

    # mnas search
    search_epochs: int
    search_lr_w: float
    search_lr_alpha: float
    search_temperature: float
    search_latency_lambda: float
    latency_budget_t0: float
    mnas_num_nodes: int
    mnas_topk_ops: int

    # federated training
    global_rounds: int
    local_epochs: int
    finetune_epochs: int
    learning_rate: float
    optimizer_name: str
    client_fraction: float

    # loss/metric
    multilabel_threshold: float
    distill_temperature: float
    distill_weight: float

    # model
    hidden_channels: int
    attention_reduction: int

    # notes
    paper_missing_fields: List[str] = field(default_factory=list)
    config_extension_notes: List[str] = field(default_factory=list)



def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



def _import_module_from_path(module_path: str, module_name: str):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot import module from {module_path}")
    module = importlib.util.module_from_spec(spec)
    # Explicitly add the module to sys.modules before executing it.
    # This helps resolve issues with dataclasses and other module-aware features
    # when dynamically loading modules.
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module



def load_runtime_config() -> MNASRuntimeConfig:
    cfg_mod = _import_module_from_path(CONFIG_PATH, "unified_fl_config")

    dataset_cfg = cfg_mod.DatasetConfig(
        dataset_name="road_multi_label",
        dataset_path=DATA_PATH,
        partitioning="non_iid_dirichlet",
        non_iid_strategy="dirichlet",
        notes=["Dataset path fixed by requirement"],
    )

    common_core = cfg_mod.CommonCoreConfig(
        dataset=dataset_cfg,
        num_rounds=100,
        batch_size=256,
        learning_rate=1e-3,
        local_epochs=1,
        optimizer="adam",
        loss_function="bce_with_logits",
        enforce_local_epochs=False,
        enforce_optimizer_and_loss=False,
    )

    merged = cfg_mod.build_unified_config(common_core=common_core, methods=[cfg_mod.MethodName.MNAS])
    mnas_cfg = merged[cfg_mod.MethodName.MNAS.value]

    ext_notes = [
        "Engineering assumption: local_epochs/global_rounds default tuned for feasibility on large ROAD dataset.",
        "Engineering assumption: use BCEWithLogitsLoss for multi-label monitoring on one-hot transformed labels.",
        "Engineering assumption: in-sample round evaluation over full data union (no hold-out split).",
        "Engineering assumption: retry + repair client partition to approximate equal size and maximize label coverage.",
        "Engineering assumption: LCSMC-Net used as classifier backbone while preserving MNAS search/federated mutual learning logic.",
    ]

    runtime = MNASRuntimeConfig(
        seed=42,
        device="cuda" if torch.cuda.is_available() else "cpu",
        data_path=DATA_PATH,
        output_dir=OUTPUT_DIR,
        scenarios=(10, 20, 50),
        dirichlet_alpha=0.5,
        min_client_samples=20,
        partition_retry_attempts=25,
        rebalance_tolerance_ratio=0.15,
        required_min_coverage_ratio=0.75,
        batch_size=int(mnas_cfg.optimization.batch_size or 256),
        num_workers=4,
        pin_memory=False,
        search_epochs=1,
        search_lr_w=1e-3,
        search_lr_alpha=1e-3,
        search_temperature=1.0,
        search_latency_lambda=2.0,
        latency_budget_t0=1.0,
        mnas_num_nodes=int(mnas_cfg.method_specific.params.get("search_space", {}).get("num_nodes", 5)),
        mnas_topk_ops=int(mnas_cfg.method_specific.params.get("search_space", {}).get("retained_paths_per_node_after_search", 2)),
        global_rounds=int(mnas_cfg.federated.num_rounds or 20),
        local_epochs=int(mnas_cfg.optimization.local_epochs or 1),
        finetune_epochs=1,
        learning_rate=float(mnas_cfg.optimization.learning_rate or 1e-3),
        optimizer_name=(mnas_cfg.optimization.optimizer or "adam").lower(),
        client_fraction=float(mnas_cfg.federated.client_fraction or 1.0),
        multilabel_threshold=0.5,
        distill_temperature=2.0,
        distill_weight=1.0,
        hidden_channels=64,
        attention_reduction=4,
        paper_missing_fields=list(mnas_cfg.paper_missing_fields),
        config_extension_notes=ext_notes,
    )

    return runtime


CFG = load_runtime_config()
set_seed(CFG.seed)
print_section("Loaded Runtime Config")
print(asdict(CFG))


[SECTION] Loaded Runtime Config
{'seed': 42, 'device': 'cpu', 'data_path': '/kaggle/input/datasets/zodasic/road-pro/Dataset/road_multi_label.csv', 'output_dir': '/kaggle/working/MNAS', 'scenarios': (10, 20, 50), 'dirichlet_alpha': 0.5, 'min_client_samples': 20, 'partition_retry_attempts': 25, 'rebalance_tolerance_ratio': 0.15, 'required_min_coverage_ratio': 0.75, 'batch_size': 256, 'num_workers': 4, 'pin_memory': False, 'search_epochs': 1, 'search_lr_w': 0.001, 'search_lr_alpha': 0.001, 'search_temperature': 1.0, 'search_latency_lambda': 2.0, 'latency_budget_t0': 1.0, 'mnas_num_nodes': 5, 'mnas_topk_ops': 2, 'global_rounds': 100, 'local_epochs': 1, 'finetune_epochs': 1, 'learning_rate': 0.001, 'optimizer_name': 'adam', 'client_fraction': 1.0, 'multilabel_threshold': 0.5, 'distill_temperature': 2.0, 'distill_weight': 1.0, 'hidden_channels': 64, 'attention_reduction': 4, 'paper_missing_fields': ['num_rounds', 'batch_size', 'learning_rate_numeric', 'local_epochs', 'optimizer', 'scheduler

## Dataset Loading + Schema Inference + Target Building

- single label column
- multiple binary label columns
- label strings dạng list/tokenized

Lưu ý: pipeline đang dùng **toàn bộ dataset để train FL**


In [5]:
def infer_label_columns(df: pd.DataFrame) -> List[str]:
    cols = list(df.columns)

    # exact/common names first
    for c in ["label", "target", "y", "labels"]:
        if c in cols:
            return [c]

    # prefix-based multi-label columns
    prefix_cols = [c for c in cols if c.lower().startswith("label_") or c.lower().startswith("target_")]
    if len(prefix_cols) >= 2:
        return prefix_cols

    # binary candidates (0/1 only) excluding obvious feature/time columns
    candidate = []
    skip = {"timestamp", "id", "can_id"}
    for c in cols:
        if c.lower() in skip:
            continue
        s = df[c]
        if pd.api.types.is_numeric_dtype(s):
            uniq = set(pd.Series(s).dropna().unique().tolist())
            if len(uniq) <= 2 and uniq.issubset({0, 1}):
                candidate.append(c)
    if len(candidate) >= 2:
        return candidate

    raise ValueError("Cannot infer label columns robustly from dataframe schema.")



def build_feature_matrix(df: pd.DataFrame, label_cols: Sequence[str]) -> Tuple[np.ndarray, List[str], StandardScaler]:
    feature_cols = [c for c in df.columns if c not in set(label_cols)]

    # keep only numeric feature columns for model input
    numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_feature_cols:
        raise ValueError("No numeric feature columns available.")

    X = df[numeric_feature_cols].to_numpy(dtype=np.float32)
    scaler = StandardScaler()
    X = scaler.fit_transform(X).astype(np.float32)

    return X, numeric_feature_cols, scaler



def build_multilabel_targets(df: pd.DataFrame, label_cols: Sequence[str]) -> Tuple[np.ndarray, List[str], np.ndarray]:
    """Return multi-hot targets and primary labels for partitioning.

    primary_labels is used for Dirichlet partition key.
    """

    # Case A: multi-column binary label indicators
    if len(label_cols) >= 2:
        y = (df[list(label_cols)].fillna(0).to_numpy() > 0).astype(np.uint8)
        label_names = list(label_cols)

        # primary label: argmax fallback, with all-zero to index 0
        row_sum = y.sum(axis=1)
        primary = y.argmax(axis=1).astype(np.int64)
        primary[row_sum == 0] = 0
        return y, label_names, primary

    # Case B: single label column
    col = label_cols[0]
    s = df[col]

    # B1: string-encoded multi-label (e.g., "1|3|7", "[1,3]")
    if s.dtype == object:
        tokens_per_row: List[List[str]] = []
        all_tokens = set()
        for val in s.fillna("").astype(str).tolist():
            vv = val.strip().replace("[", "").replace("]", "").replace(";", ",").replace("|", ",")
            toks = [t.strip() for t in vv.split(",") if t.strip() != ""]
            tokens_per_row.append(toks)
            for t in toks:
                all_tokens.add(t)

        sorted_tokens = sorted(all_tokens)
        token_to_idx = {t: i for i, t in enumerate(sorted_tokens)}
        y = np.zeros((len(tokens_per_row), len(sorted_tokens)), dtype=np.uint8)
        primary = np.zeros(len(tokens_per_row), dtype=np.int64)
        for i, toks in enumerate(tokens_per_row):
            if not toks:
                continue
            for t in toks:
                y[i, token_to_idx[t]] = 1
            primary[i] = token_to_idx[toks[0]]

        return y, [str(t) for t in sorted_tokens], primary

    # B2: standard single-label numeric/categorical => convert to one-hot (multi-label compatible)
    le = LabelEncoder()
    encoded = le.fit_transform(s.astype(str))
    n_cls = len(le.classes_)
    y = np.zeros((len(encoded), n_cls), dtype=np.uint8)
    y[np.arange(len(encoded)), encoded] = 1

    label_names = [str(x) for x in le.classes_.tolist()]
    primary = encoded.astype(np.int64)
    return y, label_names, primary



def summarize_dataset(df: pd.DataFrame, y_multi: np.ndarray, label_names: Sequence[str], feature_cols: Sequence[str]) -> Dict[str, Any]:
    label_freq = y_multi.sum(axis=0).astype(np.int64)
    label_dist = {str(label_names[i]): int(label_freq[i]) for i in range(len(label_names))}
    summary = {
        "num_samples": int(len(df)),
        "num_features": int(len(feature_cols)),
        "feature_shape": tuple([int(len(df)), int(len(feature_cols))]),
        "num_labels": int(len(label_names)),
        "label_names": list(map(str, label_names)),
        "global_label_distribution": label_dist,
    }
    return summary


print_section("Load Full Dataset")
DTYPE_MAP = {
    "timestamp": "float32",
    "ID": "int32",
    "DATA0": "int16",
    "DATA1": "int16",
    "DATA2": "int16",
    "DATA3": "int16",
    "DATA4": "int16",
    "DATA5": "int16",
    "DATA6": "int16",
    "DATA7": "int16",
    "label": "int16",
}

raw_df = pd.read_csv(CFG.data_path, dtype=DTYPE_MAP)
raw_df = raw_df.dropna().reset_index(drop=True)

label_columns = infer_label_columns(raw_df)
X_all, feature_columns, scaler_all = build_feature_matrix(raw_df, label_columns)
Y_all, label_names, primary_labels = build_multilabel_targets(raw_df, label_columns)

DATA_SUMMARY = summarize_dataset(raw_df, Y_all, label_names, feature_columns)
print_dataset_summary(DATA_SUMMARY)



[SECTION] Load Full Dataset

------------------------------------------------------------------------------------------
[STEP] Dataset Summary
------------------------------------------------------------------------------------------
- num_samples: 27415983
- num_features: 10
- feature_shape: (27415983, 10)
- num_labels: 12
- label_names: ['0', '1', '10', '11', '2', '3', '4', '5', '6', '7', '8', '9']
- global_label_distribution: {'0': 27353436, '1': 5493, '10': 8034, '11': 8034, '2': 5493, '3': 1059, '4': 43, '5': 43, '6': 11694, '7': 11694, '8': 5480, '9': 5480}


## Split Script Integration + Non-IID Dirichlet Partition (10/20/50)

Dùng logic từ `nguyen_tac_phan_chia_data_client.py` và thêm retry/repair/filter để có được:
- non-IID Dirichlet
- lọc client quá nhỏ
- gần đồng đều số mẫu
- coverage label tốt nhất có thể


In [6]:
split_mod = _import_module_from_path(SPLIT_SCRIPT_PATH, "client_split_script")
base_dirichlet_partition_indices = split_mod.dirichlet_partition_indices
base_enforce_client_constraints_indices = split_mod.enforce_client_constraints_indices



def _compute_partition_integrity(partition: List[np.ndarray], n_total: int) -> Dict[str, Any]:
    flat = np.concatenate(partition) if partition else np.array([], dtype=np.int64)
    uniq = np.unique(flat)
    return {
        "total_indices": int(len(flat)),
        "unique_indices": int(len(uniq)),
        "covers_all": bool(len(uniq) == n_total),
        "duplicate_count": int(len(flat) - len(uniq)),
    }



def _rebalance_partition_indices(
    partition: List[np.ndarray],
    labels: np.ndarray,
    tolerance_ratio: float,
    random_state: int,
) -> List[np.ndarray]:
    """Repair step to make client sizes approximately equal without data loss.

    Engineering assumption: greedy donor->receiver moves while preserving full index set.
    """

    rng = np.random.default_rng(random_state)
    labels = np.asarray(labels)

    clients = [arr.astype(np.int64).tolist() for arr in partition]
    if not clients:
        return []

    baseline = np.concatenate(partition) if partition else np.array([], dtype=np.int64)
    baseline_set = set(baseline.tolist())

    target = int(round(sum(len(c) for c in clients) / len(clients)))
    tolerance = max(1, int(target * tolerance_ratio))
    label_set_global = set(np.unique(labels).tolist())

    def _label_set(lst: List[int]) -> set:
        if not lst:
            return set()
        return set(labels[np.array(lst, dtype=np.int64)].tolist())

    for _ in range(5000):
        sizes = np.array([len(c) for c in clients], dtype=np.int64)
        donor = int(np.argmax(sizes))
        recv = int(np.argmin(sizes))

        if sizes[donor] - sizes[recv] <= tolerance:
            break

        need = max(1, (sizes[donor] - sizes[recv]) // 2)
        miss = label_set_global - _label_set(clients[recv])
        donor_pool = clients[donor]

        priority = [ix for ix in donor_pool if labels[ix] in miss]
        candidate = priority if priority else donor_pool

        move_n = min(need, len(candidate))
        selected = rng.choice(candidate, size=move_n, replace=False)
        selected_set = set(int(x) for x in selected.tolist())

        clients[donor] = [ix for ix in donor_pool if ix not in selected_set]
        clients[recv].extend(list(selected_set))

    repaired = [np.array(c, dtype=np.int64) for c in clients]

    # strict integrity check
    merged = np.concatenate(repaired) if repaired else np.array([], dtype=np.int64)
    merged_set = set(merged.tolist())
    if len(merged) != len(baseline) or merged_set != baseline_set:
        raise RuntimeError("Partition repair caused data loss/duplication.")

    for arr in repaired:
        rng.shuffle(arr)

    return repaired



def _partition_stats(partition: List[np.ndarray], y_multi: np.ndarray, label_names: Sequence[str]) -> pd.DataFrame:
    rows = []
    n_labels = y_multi.shape[1]
    for cid, idx in enumerate(partition):
        yy = y_multi[idx]
        coverage = (yy.sum(axis=0) > 0).astype(np.int64)
        coverage_count = int(coverage.sum())
        rows.append(
            {
                "client_id": int(cid),
                "num_samples": int(len(idx)),
                "coverage_count": coverage_count,
                "coverage_ratio": float(coverage_count / max(n_labels, 1)),
                "missing_labels": ",".join([str(label_names[i]) for i in np.where(coverage == 0)[0].tolist()]),
            }
        )
    return pd.DataFrame(rows)



def _score_partition(stat_df: pd.DataFrame, removed_clients_count: int) -> float:
    if len(stat_df) == 0:
        return -1e9
    sizes = stat_df["num_samples"].to_numpy(dtype=np.float64)
    mean_sz = float(sizes.mean()) if len(sizes) else 1.0
    spread = float((sizes.max() - sizes.min()) / max(mean_sz, 1.0))
    balance_score = 1.0 - spread
    coverage_score = float(stat_df["coverage_ratio"].mean())
    penalty_removed = 0.02 * float(removed_clients_count)
    return 0.55 * coverage_score + 0.45 * balance_score - penalty_removed



def build_partition_with_retry(
    labels_for_partition: np.ndarray,
    y_multi: np.ndarray,
    label_names: Sequence[str],
    num_clients: int,
    cfg: MNASRuntimeConfig,
) -> Dict[str, Any]:
    """Retry several seeds and choose best partition under constraints."""

    best_pack = None
    best_score = -1e18

    n_total = len(labels_for_partition)

    for attempt in range(cfg.partition_retry_attempts):
        seed = cfg.seed + num_clients * 1000 + attempt

        raw = base_dirichlet_partition_indices(
            labels=labels_for_partition,
            num_clients=num_clients,
            alpha=cfg.dirichlet_alpha,
            random_state=seed,
        )

        valid, removed = base_enforce_client_constraints_indices(
            raw,
            min_samples=cfg.min_client_samples,
            random_state=seed,
        )

        repaired = _rebalance_partition_indices(
            valid,
            labels=labels_for_partition,
            tolerance_ratio=cfg.rebalance_tolerance_ratio,
            random_state=seed,
        )

        integrity = _compute_partition_integrity(repaired, n_total)
        if not integrity["covers_all"]:
            continue

        stat_df = _partition_stats(repaired, y_multi=y_multi, label_names=label_names)
        score = _score_partition(stat_df, removed_clients_count=len(removed))

        if score > best_score:
            best_score = score
            best_pack = {
                "partition": repaired,
                "removed_clients": removed,
                "stats": stat_df,
                "integrity": integrity,
                "score": score,
                "seed": seed,
                "attempt": attempt,
            }

            # early stop if coverage and balance are both strong
            mean_cov = float(stat_df["coverage_ratio"].mean())
            sizes = stat_df["num_samples"].to_numpy()
            spread = float((sizes.max() - sizes.min()) / max(sizes.mean(), 1.0))
            if mean_cov >= cfg.required_min_coverage_ratio and spread <= cfg.rebalance_tolerance_ratio:
                break

    if best_pack is None:
        raise RuntimeError(f"Cannot produce valid partition for {num_clients} clients")

    return best_pack


## Federated Dataset & Dataloader (Full-data Training + In-sample Monitoring)


In [7]:
class FullTabularDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray) -> None:
        self.x = x.astype(np.float32)
        self.y = y.astype(np.uint8)

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        x = torch.from_numpy(self.x[idx]).float()
        y = torch.from_numpy(self.y[idx]).float()
        return x, y



def build_client_dataloaders(
    dataset: FullTabularDataset,
    partition: List[np.ndarray],
    batch_size: int,
    num_workers: int,
    pin_memory: bool,
) -> Dict[int, DataLoader]:
    loaders: Dict[int, DataLoader] = {}
    for cid, idx in enumerate(partition):
        loaders[cid] = DataLoader(
            Subset(dataset, idx.tolist()),
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=pin_memory,
            drop_last=False,
        )
    return loaders



def build_global_eval_dataloader(
    dataset: FullTabularDataset,
    batch_size: int,
    num_workers: int,
    pin_memory: bool,
) -> DataLoader:
    all_idx = np.arange(len(dataset), dtype=np.int64)
    return DataLoader(
        Subset(dataset, all_idx.tolist()),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
    )


## LCSMC-Net Classifier Components

Classifier dùng LCSMC-Net:
- `DepthwiseSeparableConv`
- `MultiScaleConvBlock`
- `AttentionBlock`
- `LCSMCNetClassifier`


In [8]:
class DepthwiseSeparableConv(nn.Module):
    """Depthwise separable Conv1D block used by LCSMC-style modules."""

    def __init__(self, channels: int, kernel_size: int) -> None:
        super().__init__()
        pad = kernel_size // 2
        self.dw = nn.Conv1d(channels, channels, kernel_size, padding=pad, groups=channels, bias=False)
        self.pw = nn.Conv1d(channels, channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm1d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.dw(x)
        x = self.pw(x)
        x = self.bn(x)
        return F.silu(x)


class AttentionBlock(nn.Module):
    """Lightweight channel-temporal attention (engineering reconstruction of LCTA-style idea)."""

    def __init__(self, channels: int, reduction: int = 4) -> None:
        super().__init__()
        hidden = max(1, channels // reduction)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)
        self.temporal = nn.Conv1d(channels, 1, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # channel attention
        c = x.mean(dim=-1)
        c = F.silu(self.fc1(c))
        c = torch.sigmoid(self.fc2(c)).unsqueeze(-1)

        # temporal attention
        t = torch.sigmoid(self.temporal(x))
        return x * c * t


class MultiScaleConvBlock(nn.Module):
    """Separable multiscale block in LCSMC style."""

    def __init__(self, channels: int, kernels: Sequence[int] = (3, 5, 7), reduction: int = 4) -> None:
        super().__init__()
        self.branches = nn.ModuleList([DepthwiseSeparableConv(channels, k) for k in kernels])
        self.merge = nn.Conv1d(channels * len(kernels), channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm1d(channels)
        self.attn = AttentionBlock(channels, reduction=reduction)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        outs = [b(x) for b in self.branches]
        y = torch.cat(outs, dim=1)
        y = self.merge(y)
        y = self.bn(y)
        y = F.silu(y)
        y = self.attn(y)
        return y + x


class LCSMCNetClassifier(nn.Module):
    """Baseline LCSMC classifier for multi-label outputs."""

    def __init__(self, input_dim: int, num_labels: int, hidden_channels: int = 64, num_blocks: int = 3, reduction: int = 4) -> None:
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Conv1d(1, hidden_channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(hidden_channels),
            nn.SiLU(),
        )
        self.blocks = nn.ModuleList([MultiScaleConvBlock(hidden_channels, kernels=(3, 5, 7), reduction=reduction) for _ in range(num_blocks)])
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(hidden_channels, hidden_channels),
            nn.SiLU(),
            nn.Linear(hidden_channels, num_labels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim == 2:
            x = x.unsqueeze(1)  # [B, 1, F]
        y = self.input_proj(x)
        for block in self.blocks:
            y = block(y)
        logits = self.head(y)
        return logits


## MNAS Search on top of LCSMC Backbone

- compressed search space (shared large kernel)
- single-path search via gumbel hard routing
- hardware-aware latency regularization


In [9]:
OPS = ["conv_7", "conv_5", "conv_3", "conv_1", "max_pool", "avg_pool", "skip", "identity"]


class SharedKernelConv1d(nn.Module):
    """Compressed search space: one max-kernel tensor reused by smaller kernels."""

    def __init__(self, channels: int, max_kernel: int = 7) -> None:
        super().__init__()
        assert max_kernel % 2 == 1
        self.max_kernel = max_kernel
        self.dw = nn.Conv1d(channels, channels, kernel_size=max_kernel, padding=max_kernel // 2, groups=channels, bias=False)
        self.pw = nn.Conv1d(channels, channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm1d(channels)

    def forward(self, x: torch.Tensor, kernel_size: int) -> torch.Tensor:
        if kernel_size == self.max_kernel:
            y = self.dw(x)
        else:
            w = self.dw.weight
            center = self.max_kernel // 2
            half = kernel_size // 2
            ws = w[:, :, center - half:center + half + 1]
            y = F.conv1d(x, ws, bias=None, stride=1, padding=half, groups=x.shape[1])
        y = self.pw(y)
        y = self.bn(y)
        return F.silu(y)


class SearchOpBank(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.shared_conv = SharedKernelConv1d(channels, max_kernel=7)
        self.proj_pool = nn.Conv1d(channels, channels, kernel_size=1, bias=False)

    def forward(self, x: torch.Tensor, op_name: str) -> torch.Tensor:
        if op_name == "conv_7":
            return self.shared_conv(x, 7)
        if op_name == "conv_5":
            return self.shared_conv(x, 5)
        if op_name == "conv_3":
            return self.shared_conv(x, 3)
        if op_name == "conv_1":
            return self.shared_conv(x, 1)
        if op_name == "max_pool":
            return self.proj_pool(F.max_pool1d(x, kernel_size=3, stride=1, padding=1))
        if op_name == "avg_pool":
            return self.proj_pool(F.avg_pool1d(x, kernel_size=3, stride=1, padding=1))
        if op_name == "skip":
            return x
        if op_name == "identity":
            return x
        raise KeyError(op_name)


class SearchNode(nn.Module):
    """Single-path searchable node.

    Engineering assumption:
    - use hard gumbel routing to approximate binary path activation.
    """

    def __init__(self, channels: int, op_names: Sequence[str]) -> None:
        super().__init__()
        self.op_names = list(op_names)
        self.bank = SearchOpBank(channels)
        self.alpha = nn.Parameter(torch.zeros(len(self.op_names), dtype=torch.float32))

    def forward(self, x: torch.Tensor, temperature: float = 1.0) -> Tuple[torch.Tensor, torch.Tensor]:
        gate = F.gumbel_softmax(self.alpha, tau=temperature, hard=True)
        idx = int(torch.argmax(gate).item())
        op = self.op_names[idx]
        y = self.bank(x, op)
        # retain differentiability wrt alpha through gate scalar
        y = y * gate[idx]
        return y, gate

    def topk_ops(self, k: int) -> List[str]:
        probs = F.softmax(self.alpha.detach(), dim=0)
        top_idx = torch.topk(probs, k=min(k, len(self.op_names))).indices.tolist()
        return [self.op_names[i] for i in top_idx]


class SearchableLCSMCSupernet(nn.Module):
    """LCSMC-based supernet for MNAS search."""

    def __init__(self, input_dim: int, num_labels: int, channels: int, num_nodes: int, reduction: int, op_cost: Dict[str, float]):
        super().__init__()
        self.channels = channels
        self.num_nodes = num_nodes
        self.op_cost = dict(op_cost)

        self.stem = nn.Sequential(
            nn.Conv1d(1, channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(channels),
            nn.SiLU(),
        )

        self.nodes = nn.ModuleList([SearchNode(channels, OPS) for _ in range(num_nodes)])
        self.post_attn = AttentionBlock(channels, reduction=reduction)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(channels, channels),
            nn.SiLU(),
            nn.Linear(channels, num_labels),
        )

    def forward(self, x: torch.Tensor, temperature: float = 1.0) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        if x.ndim == 2:
            x = x.unsqueeze(1)
        y = self.stem(x)
        gates = []
        for node in self.nodes:
            out, gate = node(y, temperature=temperature)
            y = y + out
            gates.append(gate)
        y = self.post_attn(y)
        logits = self.head(y)
        return logits, gates

    def architecture_parameters(self) -> List[nn.Parameter]:
        return [n.alpha for n in self.nodes]

    def weight_parameters(self) -> List[nn.Parameter]:
        arch_ids = {id(p) for p in self.architecture_parameters()}
        return [p for p in self.parameters() if id(p) not in arch_ids]

    def expected_latency(self) -> torch.Tensor:
        lat = torch.tensor(0.0, device=next(self.parameters()).device)
        for node in self.nodes:
            probs = F.softmax(node.alpha, dim=0)
            op_lat = torch.tensor([self.op_cost[o] for o in node.op_names], dtype=torch.float32, device=probs.device)
            lat = lat + (probs * op_lat).sum()
        return lat

    def export_topk_architecture(self, k: int) -> List[List[str]]:
        return [node.topk_ops(k) for node in self.nodes]


class PersonalizedSearchedLCSMC(nn.Module):
    """Personalized model constructed from searched top-k ops per node."""

    def __init__(self, input_dim: int, num_labels: int, channels: int, reduction: int, selected_ops: List[List[str]]) -> None:
        super().__init__()
        self.selected_ops = selected_ops
        self.stem = nn.Sequential(
            nn.Conv1d(1, channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(channels),
            nn.SiLU(),
        )
        self.banks = nn.ModuleList([SearchOpBank(channels) for _ in selected_ops])
        self.post_attn = AttentionBlock(channels, reduction=reduction)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(channels, channels),
            nn.SiLU(),
            nn.Linear(channels, num_labels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim == 2:
            x = x.unsqueeze(1)
        y = self.stem(x)
        for bank, ops in zip(self.banks, self.selected_ops):
            # engineering assumption: average top-k op outputs
            outs = [bank(y, op) for op in ops]
            mix = torch.stack(outs, dim=0).mean(dim=0)
            y = y + mix
        y = self.post_attn(y)
        return self.head(y)


class ProxyLCSMC(nn.Module):
    """Unified proxy model with fixed architecture for federated aggregation."""

    def __init__(self, input_dim: int, num_labels: int, channels: int, reduction: int) -> None:
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Conv1d(1, channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(channels),
            nn.SiLU(),
        )
        self.block1 = DepthwiseSeparableConv(channels, kernel_size=5)
        self.block2 = DepthwiseSeparableConv(channels, kernel_size=5)
        self.block3 = DepthwiseSeparableConv(channels, kernel_size=5)
        self.attn = AttentionBlock(channels, reduction=reduction)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(channels, channels),
            nn.SiLU(),
            nn.Linear(channels, num_labels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim == 2:
            x = x.unsqueeze(1)
        y = self.input_proj(x)
        y = self.block1(y)
        y = self.block2(y)
        y = self.block3(y)
        y = self.attn(y)
        return self.head(y)



def search_personalized_architecture(
    train_loader: DataLoader,
    input_dim: int,
    num_labels: int,
    cfg: MNASRuntimeConfig,
    op_cost: Dict[str, float],
    device: torch.device,
) -> List[List[str]]:
    """Run MNAS search and return top-k operations per node."""

    model = SearchableLCSMCSupernet(
        input_dim=input_dim,
        num_labels=num_labels,
        channels=cfg.hidden_channels,
        num_nodes=cfg.mnas_num_nodes,
        reduction=cfg.attention_reduction,
        op_cost=op_cost,
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    opt_w = torch.optim.Adam(model.weight_parameters(), lr=cfg.search_lr_w)
    opt_a = torch.optim.Adam(model.architecture_parameters(), lr=cfg.search_lr_alpha)

    model.train()
    for _ in range(cfg.search_epochs):
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            opt_w.zero_grad(set_to_none=True)
            opt_a.zero_grad(set_to_none=True)

            logits, _ = model(xb, temperature=cfg.search_temperature)
            label_loss = criterion(logits, yb)

            # hardware-aware penalty from expected latency
            latency = model.expected_latency()
            latency_penalty = (latency / max(cfg.latency_budget_t0, 1e-8)) ** cfg.search_latency_lambda
            loss = label_loss + latency_penalty

            loss.backward()
            opt_w.step()
            opt_a.step()

    selected = model.export_topk_architecture(k=cfg.mnas_topk_ops)
    return selected


## Adaptive Federated Mutual Learning (MNAS core)

- Mỗi client giữ `personalized model` + `proxy model`
- Mutual distillation hai chiều, adaptive scaling theo paper
- Server chỉ aggregate proxy weights bằng FedAvg


In [10]:
@dataclass
class MNASClientState:
    client_id: int
    train_loader: DataLoader
    num_samples: int
    selected_ops: List[List[str]]
    personalized_model: nn.Module
    proxy_model: nn.Module
    opt_personalized: torch.optim.Optimizer
    opt_proxy: torch.optim.Optimizer
    label_coverage: List[int]



def _build_optimizer(model: nn.Module, cfg: MNASRuntimeConfig) -> torch.optim.Optimizer:
    name = cfg.optimizer_name.lower()
    if name == "sgd":
        return torch.optim.SGD(model.parameters(), lr=cfg.learning_rate, momentum=0.9)
    if name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate)
    return torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)



def _adaptive_distillation_loss(
    logits_personal: torch.Tensor,
    logits_proxy: torch.Tensor,
    label_loss_personal: torch.Tensor,
    label_loss_proxy: torch.Tensor,
    temperature: float,
    eps: float = 1e-6,
) -> torch.Tensor:
    ps = torch.sigmoid(logits_personal / temperature)
    pr = torch.sigmoid(logits_proxy / temperature)
    dist = F.mse_loss(ps, pr)
    adapt = dist / (label_loss_personal.detach() + label_loss_proxy.detach() + eps)
    return adapt



def train_one_local_epoch_mutual(
    client: MNASClientState,
    cfg: MNASRuntimeConfig,
    device: torch.device,
) -> Dict[str, float]:
    criterion = nn.BCEWithLogitsLoss()
    client.personalized_model.train()
    client.proxy_model.train()

    total_personal = 0.0
    total_proxy = 0.0
    total_n = 0

    for xb, yb in client.train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        client.opt_personalized.zero_grad(set_to_none=True)
        client.opt_proxy.zero_grad(set_to_none=True)

        logits_p = client.personalized_model(xb)
        logits_r = client.proxy_model(xb)

        loss_p = criterion(logits_p, yb)
        loss_r = criterion(logits_r, yb)
        dist_w = _adaptive_distillation_loss(
            logits_personal=logits_p,
            logits_proxy=logits_r,
            label_loss_personal=loss_p,
            label_loss_proxy=loss_r,
            temperature=cfg.distill_temperature,
        )

        total_loss_p = loss_p + cfg.distill_weight * dist_w
        total_loss_r = loss_r + cfg.distill_weight * dist_w

        total_loss_p.backward(retain_graph=True)
        total_loss_r.backward()

        client.opt_personalized.step()
        client.opt_proxy.step()

        bs = xb.size(0)
        total_personal += float(total_loss_p.item()) * bs
        total_proxy += float(total_loss_r.item()) * bs
        total_n += bs

    return {
        "loss_personal": total_personal / max(total_n, 1),
        "loss_proxy": total_proxy / max(total_n, 1),
    }



def fine_tune_personalized_model(
    client: MNASClientState,
    cfg: MNASRuntimeConfig,
    device: torch.device,
) -> None:
    criterion = nn.BCEWithLogitsLoss()
    client.personalized_model.train()

    for _ in range(cfg.finetune_epochs):
        for xb, yb in client.train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            client.opt_personalized.zero_grad(set_to_none=True)
            logits = client.personalized_model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            client.opt_personalized.step()



def fedavg_proxy_states(states: List[Tuple[Dict[str, torch.Tensor], int]]) -> Dict[str, torch.Tensor]:
    """Aggregate proxy model states by sample-weighted FedAvg."""

    total = float(sum(n for _, n in states))
    if total <= 0:
        raise ValueError("Invalid total sample count for FedAvg")

    keys = list(states[0][0].keys())
    out = {k: torch.zeros_like(states[0][0][k], dtype=torch.float32) for k in keys}

    for state_dict, n in states:
        w = float(n) / total
        for k in keys:
            out[k] += state_dict[k].float() * w

    return out


## Metrics, Sanity Checks, Evaluation

Evaluate metric theo round:
- loss
- precision
- recall
- F1


In [11]:
def sanity_check_shapes(
    x: np.ndarray,
    y: np.ndarray,
    num_labels: int,
    feature_dim: int,
) -> None:
    assert x.ndim == 2, f"Expected X rank-2, got {x.ndim}"
    assert y.ndim == 2, f"Expected Y rank-2, got {y.ndim}"
    assert x.shape[0] == y.shape[0], "X/Y sample mismatch"
    assert x.shape[1] == feature_dim, f"Feature dim mismatch: {x.shape[1]} != {feature_dim}"
    assert y.shape[1] == num_labels, f"Label dim mismatch: {y.shape[1]} != {num_labels}"


@torch.no_grad()
def evaluate_global_model(
    model: nn.Module,
    global_eval_loader: DataLoader,
    threshold: float,
    device: torch.device,
) -> Dict[str, float]:
    criterion = nn.BCEWithLogitsLoss()
    model.eval()

    all_probs = []
    all_targets = []
    running_loss = 0.0
    n_total = 0

    for xb, yb in global_eval_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        probs = torch.sigmoid(logits)
        all_probs.append(probs.detach().cpu().numpy())
        all_targets.append(yb.detach().cpu().numpy())

        bs = xb.size(0)
        running_loss += float(loss.item()) * bs
        n_total += bs

    probs_all = np.concatenate(all_probs, axis=0)
    y_true = np.concatenate(all_targets, axis=0).astype(np.int64)
    y_pred = (probs_all >= threshold).astype(np.int64)

    p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="micro", zero_division=0
    )
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    return {
        "loss": float(running_loss / max(n_total, 1)),
        "precision": float(p_macro),
        "recall": float(r_macro),
        "f1": float(f_macro),
        "precision_micro": float(p_micro),
        "recall_micro": float(r_micro),
        "f1_micro": float(f_micro),
    }



def get_label_coverage_for_client(indices: np.ndarray, y_multi: np.ndarray) -> List[int]:
    yy = y_multi[indices]
    return np.where(yy.sum(axis=0) > 0)[0].astype(int).tolist()


## Full MNAS Federated Pipeline (10/20/50)


In [12]:
def _build_operation_cost_table() -> Dict[str, float]:
    """Hardware-aware surrogate op cost table (paper-aligned concept)."""
    return {
        "conv_7": 2.9,
        "conv_5": 2.2,
        "conv_3": 1.6,
        "conv_1": 1.0,
        "max_pool": 0.35,
        "avg_pool": 0.35,
        "skip": 0.08,
        "identity": 0.05,
    }



def run_mnas_experiment_for_n_clients(
    num_clients: int,
    x_all: np.ndarray,
    y_all: np.ndarray,
    primary_labels_local: np.ndarray,
    label_names_local: Sequence[str],
    cfg: MNASRuntimeConfig,
    show_progress: bool = True,
    client_progress_interval: int = 5,
) -> Dict[str, Any]:
    t_start = time.time()

    if show_progress:
        print_section(f"MNAS Experiment | {num_clients} clients")

    part_pack = build_partition_with_retry(
        labels_for_partition=primary_labels_local,
        y_multi=y_all,
        label_names=label_names_local,
        num_clients=num_clients,
        cfg=cfg,
    )
    partition = part_pack["partition"]
    stat_df = part_pack["stats"]
    integrity = part_pack["integrity"]

    if show_progress:
        print_client_split_summary(num_clients, stat_df, {
            "best_seed": part_pack["seed"],
            "best_attempt": part_pack["attempt"],
            "partition_score": part_pack["score"],
            "integrity": integrity,
            "removed_clients": part_pack["removed_clients"],
        })

    dataset = FullTabularDataset(x_all, y_all)
    client_loaders = build_client_dataloaders(
        dataset,
        partition,
        batch_size=cfg.batch_size,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
    )
    global_eval_loader = build_global_eval_dataloader(
        dataset,
        batch_size=cfg.batch_size,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
    )

    input_dim = x_all.shape[1]
    num_labels = y_all.shape[1]
    sanity_check_shapes(x_all, y_all, num_labels=num_labels, feature_dim=input_dim)

    op_cost = _build_operation_cost_table()

    # Step 1: Search personalized architecture per client
    client_states: Dict[int, MNASClientState] = {}
    device = torch.device(cfg.device)

    if show_progress:
        print_subsection("Architecture Search (MNAS)")

    for cpos, (cid, loader) in enumerate(client_loaders.items(), 1):
        selected_ops = search_personalized_architecture(
            train_loader=loader,
            input_dim=input_dim,
            num_labels=num_labels,
            cfg=cfg,
            op_cost=op_cost,
            device=device,
        )

        personalized = PersonalizedSearchedLCSMC(
            input_dim=input_dim,
            num_labels=num_labels,
            channels=cfg.hidden_channels,
            reduction=cfg.attention_reduction,
            selected_ops=selected_ops,
        ).to(device)

        proxy = ProxyLCSMC(
            input_dim=input_dim,
            num_labels=num_labels,
            channels=cfg.hidden_channels,
            reduction=cfg.attention_reduction,
        ).to(device)

        opt_p = _build_optimizer(personalized, cfg)
        opt_r = _build_optimizer(proxy, cfg)

        coverage = get_label_coverage_for_client(partition[cid], y_all)

        client_states[cid] = MNASClientState(
            client_id=cid,
            train_loader=loader,
            num_samples=int(len(partition[cid])),
            selected_ops=selected_ops,
            personalized_model=personalized,
            proxy_model=proxy,
            opt_personalized=opt_p,
            opt_proxy=opt_r,
            label_coverage=coverage,
        )

        if show_progress and (cpos % max(1, client_progress_interval) == 0 or cpos == len(client_loaders)):
            print(f"search_progress={cpos}/{len(client_loaders)} | client_id={cid} | selected_ops={selected_ops}")

    # initialize server global proxy with first client
    first_cid = sorted(client_states.keys())[0]
    global_proxy_state = copy.deepcopy(client_states[first_cid].proxy_model.state_dict())

    # Federated rounds
    round_rows: List[Dict[str, Any]] = []

    selected_client_ids = sorted(client_states.keys())
    if cfg.client_fraction < 1.0:
        rng = np.random.default_rng(cfg.seed + num_clients)

    for rnd in range(1, cfg.global_rounds + 1):
        if cfg.client_fraction >= 1.0:
            active_ids = selected_client_ids
        else:
            m = max(1, int(len(selected_client_ids) * cfg.client_fraction))
            active_ids = sorted(np.random.choice(selected_client_ids, size=m, replace=False).tolist())

        if show_progress:
            print_round_start(rnd, cfg.global_rounds, len(active_ids))

        # broadcast global proxy
        for cid in active_ids:
            client_states[cid].proxy_model.load_state_dict(global_proxy_state, strict=True)

        uploaded_states: List[Tuple[Dict[str, torch.Tensor], int]] = []
        personal_losses = []
        proxy_losses = []

        # local update
        for pos, cid in enumerate(active_ids, 1):
            st = client_states[cid]
            local_loss_personal = 0.0
            local_loss_proxy = 0.0
            for _ in range(cfg.local_epochs):
                out = train_one_local_epoch_mutual(st, cfg=cfg, device=device)
                local_loss_personal = out["loss_personal"]
                local_loss_proxy = out["loss_proxy"]

            personal_losses.append(local_loss_personal)
            proxy_losses.append(local_loss_proxy)

            uploaded_states.append((copy.deepcopy(st.proxy_model.state_dict()), st.num_samples))

            if show_progress and (pos % max(1, client_progress_interval) == 0 or pos == len(active_ids)):
                print_client_training_progress(
                    round_idx=rnd,
                    client_pos=pos,
                    total_clients=len(active_ids),
                    client_id=cid,
                    mode="mutual_local_update",
                    loss_personal=local_loss_personal,
                    loss_proxy=local_loss_proxy,
                )

        # server aggregate proxy
        global_proxy_state = fedavg_proxy_states(uploaded_states)

        # in-sample monitoring metric on global data union
        eval_proxy = ProxyLCSMC(
            input_dim=input_dim,
            num_labels=num_labels,
            channels=cfg.hidden_channels,
            reduction=cfg.attention_reduction,
        ).to(device)
        eval_proxy.load_state_dict(global_proxy_state, strict=True)

        metrics = evaluate_global_model(
            model=eval_proxy,
            global_eval_loader=global_eval_loader,
            threshold=cfg.multilabel_threshold,
            device=device,
        )

        round_rows.append(
            {
                "num_clients": num_clients,
                "round": rnd,
                "loss": metrics["loss"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
                "precision_micro": metrics["precision_micro"],
                "recall_micro": metrics["recall_micro"],
                "f1_micro": metrics["f1_micro"],
                "mean_local_loss_personal": float(np.mean(personal_losses) if personal_losses else 0.0),
                "mean_local_loss_proxy": float(np.mean(proxy_losses) if proxy_losses else 0.0),
                "active_clients": len(active_ids),
            }
        )

        if show_progress:
            print_round_metrics(rnd, metrics)

    # optional fine-tuning personalized models at end (paper-aligned step)
    if show_progress:
        print_subsection("Final Personalized Fine-tuning")
    for cid in sorted(client_states.keys()):
        fine_tune_personalized_model(client_states[cid], cfg=cfg, device=device)

    history_df = pd.DataFrame(round_rows)
    final_metrics = history_df.iloc[-1][["loss", "precision", "recall", "f1", "precision_micro", "recall_micro", "f1_micro"]].to_dict()

    elapsed = time.time() - t_start
    if show_progress:
        print_experiment_summary(num_clients=num_clients, final_metrics=final_metrics, elapsed_sec=elapsed)

    # save scenario artifacts
    mdir = OUT / "outputs" / "metrics"
    fdir = OUT / "outputs" / "figures"
    cdir = OUT / "outputs" / "checkpoints"
    mdir.mkdir(parents=True, exist_ok=True)
    fdir.mkdir(parents=True, exist_ok=True)
    cdir.mkdir(parents=True, exist_ok=True)

    hist_csv = mdir / f"mnas_round_metrics_{num_clients}clients.csv"
    hist_json = mdir / f"mnas_round_metrics_{num_clients}clients.json"
    split_csv = mdir / f"mnas_client_split_{num_clients}clients.csv"
    arch_json = mdir / f"mnas_selected_architecture_{num_clients}clients.json"
    ckpt_path = cdir / f"mnas_global_proxy_{num_clients}clients.pt"

    history_df.to_csv(hist_csv, index=False)
    history_df.to_json(hist_json, orient="records", indent=2)
    stat_df.to_csv(split_csv, index=False)

    arch_export = {str(cid): client_states[cid].selected_ops for cid in sorted(client_states.keys())}
    with open(arch_json, "w", encoding="utf-8") as f:
        json.dump(arch_export, f, indent=2, ensure_ascii=False)

    torch.save({"global_proxy_state": global_proxy_state, "num_clients": num_clients}, ckpt_path)

    return {
        "num_clients": num_clients,
        "history_df": history_df,
        "split_stats": stat_df,
        "selected_arch": arch_export,
        "final_metrics": final_metrics,
        "elapsed_sec": elapsed,
        "saved_files": {
            "history_csv": str(hist_csv),
            "history_json": str(hist_json),
            "split_csv": str(split_csv),
            "arch_json": str(arch_json),
            "checkpoint": str(ckpt_path),
        },
    }



def run_all_mnas_scenarios(
    x_all: np.ndarray,
    y_all: np.ndarray,
    primary_labels_local: np.ndarray,
    label_names_local: Sequence[str],
    cfg: MNASRuntimeConfig,
    show_progress: bool = True,
    client_progress_interval: int = 5,
) -> Dict[int, Dict[str, Any]]:
    results: Dict[int, Dict[str, Any]] = {}
    for n_clients in cfg.scenarios:
        results[int(n_clients)] = run_mnas_experiment_for_n_clients(
            num_clients=int(n_clients),
            x_all=x_all,
            y_all=y_all,
            primary_labels_local=primary_labels_local,
            label_names_local=label_names_local,
            cfg=cfg,
            show_progress=show_progress,
            client_progress_interval=client_progress_interval,
        )
    return results


## Section 13 — Visualization

In [13]:
def plot_round_metrics(results: Dict[int, Dict[str, Any]], out_dir: Path) -> Dict[str, str]:
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = {}

    metric_names = ["loss", "precision", "recall", "f1"]
    for metric in metric_names:
        plt.figure(figsize=(10, 5))
        for n_clients, pack in sorted(results.items()):
            df = pack["history_df"]
            if metric in df.columns:
                plt.plot(df["round"], df[metric], marker="o", label=f"{n_clients} clients")
        plt.title(f"{metric} vs round")
        plt.xlabel("round")
        plt.ylabel(metric)
        plt.grid(alpha=0.3)
        plt.legend()
        out = out_dir / f"mnas_{metric}_curve_10_20_50.png"
        plt.tight_layout()
        plt.savefig(out, dpi=170)
        plt.close()
        saved[metric] = str(out)

    return saved



def save_results_summary(results: Dict[int, Dict[str, Any]], out_dir: Path) -> Dict[str, str]:
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for n_clients, pack in sorted(results.items()):
        row = {"num_clients": int(n_clients), "elapsed_sec": float(pack["elapsed_sec"])}
        row.update({k: float(v) for k, v in pack["final_metrics"].items()})
        rows.append(row)

    df = pd.DataFrame(rows)
    csv_path = out_dir / "mnas_final_summary_10_20_50.csv"
    json_path = out_dir / "mnas_final_summary_10_20_50.json"

    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient="records", indent=2)

    return {"summary_csv": str(csv_path), "summary_json": str(json_path)}


In [ ]:
RUN_TRAINING = True


def main_pipeline_no_autorun() -> None:
    print_section("Main Pipeline")
    print("RUN_TRAINING=False -> notebook will not run federated training automatically.")
    print("When you want to run on A100, set RUN_TRAINING=True and execute the cell manually.")


if RUN_TRAINING:
    results = run_all_mnas_scenarios(
        x_all=X_all,
        y_all=Y_all,
        primary_labels_local=primary_labels,
        label_names_local=label_names,
        cfg=CFG,
        show_progress=True,
        client_progress_interval=5,
    )

    fig_paths = plot_round_metrics(results, out_dir=OUT / "outputs" / "figures")
    summary_paths = save_results_summary(results, out_dir=OUT / "outputs" / "metrics")

    print("Saved figures:", fig_paths)
    print("Saved summaries:", summary_paths)
    print("Saved explanation PDF:", pdf_path)
else:
    main_pipeline_no_autorun()



[SECTION] MNAS Experiment | 10 clients


## XAI — Right-for-the-Right-Reasons (RRR) Saliency Integration

Tích hợp XAI (Ross et al., 2017) vào pipeline MNAS/LCSMC:
- **Expert mask** theo tri thức miền cho ROAD dataset
- **SaliencyLearningLoss** (BCE multi-label + gradient penalty)
- **compute_saliency_map** cho inference / visualization
- **SaliencyVisualizer** để vẽ bar chart saliency theo feature


In [ ]:
import torch.autograd as autograd
import torch.nn.functional as F

# ─────────────────────────────────────────────────────────────────────────────
# A. Expert Binary Mask — ROAD dataset (tri thức miền)
# ─────────────────────────────────────────────────────────────────────────────

def build_road_expert_mask(feature_columns, penalized_prefixes=("DATA",), device=None):
    """Tạo expert binary mask cho ROAD dataset.

    Rules (tri thức miền):
      - timestamp / time → penalize (mask=1): artifact ghi log, không phản ánh tấn công.
      - DATA0–DATA7 (payload) → penalize (mask=1): nơi tấn công can thiệp payload.
      - ID (CAN frame ID) → allow (mask=0): ID tương quan với tấn công masquerade.

    Returns: Tensor (num_features,), float32, 0.0 hoặc 1.0.
    """
    vals = []
    for col in feature_columns:
        up = col.upper()
        penalize = any(up.startswith(p.upper()) for p in penalized_prefixes)
        if up in ("TIMESTAMP", "TIME"):
            penalize = True
        vals.append(1.0 if penalize else 0.0)
    mask = torch.tensor(vals, dtype=torch.float32)
    if device is not None:
        mask = mask.to(device)
    return mask


# ─────────────────────────────────────────────────────────────────────────────
# B. SaliencyLearningLoss — RRR cho multi-label (BCEWithLogitsLoss)
# ─────────────────────────────────────────────────────────────────────────────

class SaliencyLearningLoss(nn.Module):
    """Hàm loss RRR cho bài toán multi-label.

    total_loss = BCEWithLogitsLoss(logits, targets)
               + lambda_saliency * sum( (∂score/∂x * expert_mask)^2 )

    score = sum( log_sigmoid(logits) * targets )   # multi-hot weighted log-prob

    Ref: Ross et al. (2017) https://arxiv.org/abs/1703.03717
    """

    def __init__(self, lambda_saliency: float = 1.0):
        super().__init__()
        self.base_criterion = nn.BCEWithLogitsLoss()
        self.lambda_saliency = lambda_saliency

    def forward(self, model, x, targets, expert_mask):
        """
        Args:
            model       : LCSMC / Personalized / ProxyLCSMC model.
            x           : (batch, num_features) float tensor.
            targets     : (batch, num_labels)   float multi-hot tensor.
            expert_mask : (num_features,) hoặc (batch, num_features).
                          1 = feature bị phạt, 0 = feature được phép học.
        Returns:
            total_loss, base_loss.detach(), saliency_penalty.detach()
        """
        x_probe = x.detach().requires_grad_(True)
        logits = model(x_probe)

        # Base loss (BCE multi-label)
        base_loss = self.base_criterion(logits, targets)

        # Score để tính gradient: log-sigmoid weighted by multi-hot targets
        log_p = F.logsigmoid(logits)        # (batch, num_labels)
        score = (log_p * targets).sum()     # scalar

        grads = autograd.grad(
            outputs=score, inputs=x_probe,
            create_graph=True, retain_graph=True, only_inputs=True
        )[0]                                # (batch, num_features)

        # Broadcast mask
        mask = expert_mask.to(x_probe.device)
        if mask.dim() == 1:
            mask = mask.unsqueeze(0)        # (1, num_features)

        saliency_penalty = torch.sum((grads * mask) ** 2)
        total_loss = base_loss + self.lambda_saliency * saliency_penalty
        return total_loss, base_loss.detach(), saliency_penalty.detach()


# ─────────────────────────────────────────────────────────────────────────────
# C. compute_saliency_map — inference / visualization
# ─────────────────────────────────────────────────────────────────────────────

@torch.enable_grad()
def compute_saliency_map(model, x, target_label_indices=None):
    """Gradient-based saliency map (không ảnh hưởng training).

    Returns: saliency (batch, features) = |∂score/∂x_i|
    """
    was_training = model.training
    model.eval()

    x_probe = x.detach().requires_grad_(True)
    logits = model(x_probe)
    log_p = F.logsigmoid(logits)

    if target_label_indices is None:
        score = log_p.sum()
    else:
        idx = target_label_indices.to(logits.device)
        if idx.dim() == 1:
            idx = idx.unsqueeze(1)
        score = log_p.gather(1, idx).sum()

    score.backward()
    saliency = x_probe.grad.abs().detach()

    if was_training:
        model.train()
    return saliency


# ─────────────────────────────────────────────────────────────────────────────
# D. SaliencyVisualizer
# ─────────────────────────────────────────────────────────────────────────────

class SaliencyVisualizer:
    """Tổng hợp và vẽ saliency map trên DataLoader."""

    def __init__(self, model, feature_columns, device):
        self.model = model
        self.feature_columns = list(feature_columns)
        self.device = device

    def aggregate(self, data_loader, max_batches=100, target_label_indices=None):
        all_sal = []
        for i, (xb, _) in enumerate(data_loader):
            if i >= max_batches:
                break
            sal = compute_saliency_map(self.model, xb.to(self.device), target_label_indices)
            all_sal.append(sal.cpu().numpy())
        if not all_sal:
            raise ValueError("DataLoader rỗng.")
        stacked = np.concatenate(all_sal, axis=0)
        return {
            "mean_saliency": stacked.mean(axis=0),
            "std_saliency":  stacked.std(axis=0),
            "feature_names": self.feature_columns,
        }

    def plot_bar(self, stats, title="Feature Saliency", expert_mask=None, save_path=None):
        import matplotlib.pyplot as plt
        from matplotlib.patches import Patch

        mean_sal = stats["mean_saliency"]
        std_sal  = stats["std_saliency"]
        names    = stats["feature_names"]
        x_pos    = np.arange(len(names))

        colors = ["steelblue"] * len(names)
        if expert_mask is not None:
            m = expert_mask.cpu().numpy()
            for i, v in enumerate(m):
                if v > 0.5:
                    colors[i] = "tomato"

        fig, ax = plt.subplots(figsize=(max(8, len(names) * 0.8), 5))
        ax.bar(x_pos, mean_sal, yerr=std_sal, color=colors, capsize=4, alpha=0.85)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(names, rotation=45, ha="right", fontsize=9)
        ax.set_ylabel("|∂score/∂x|  (mean ± std)")
        ax.set_title(title)
        ax.grid(axis="y", alpha=0.3)
        ax.legend(handles=[
            Patch(facecolor="steelblue", label="Allowed (mask=0)"),
            Patch(facecolor="tomato",    label="Penalized (mask=1)"),
        ], loc="upper right")
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150)
        plt.show()
        plt.close(fig)


print("XAI components loaded: SaliencyLearningLoss, build_road_expert_mask, compute_saliency_map, SaliencyVisualizer")


In [ ]:
# ─── Build expert mask (feature_columns đã có từ build_feature_matrix) ───────
EXPERT_MASK = build_road_expert_mask(
    feature_columns=feature_columns,
    penalized_prefixes=("DATA",),   # DATA0–DATA7 bị phạt; ID được phép
)
print("Expert mask (1=penalized, 0=allowed):")
for name, val in zip(feature_columns, EXPERT_MASK.tolist()):
    print(f"  {name:12s} → {'PENALIZED' if val > 0.5 else 'allowed'}")

# ─── Khởi tạo XAI loss ────────────────────────────────────────────────────
XAI_LAMBDA    = 0.05      # Điều chỉnh nhỏ để không cản hội tụ lúc đầu
XAI_CRITERION = SaliencyLearningLoss(lambda_saliency=XAI_LAMBDA)
print(f"\nSaliencyLearningLoss ready (lambda_saliency={XAI_LAMBDA})")


## XAI — Training Loop tích hợp Saliency Penalty

`train_one_local_epoch_mutual_xai` là phiên bản mở rộng dùng `SaliencyLearningLoss`
cho **personalized model**; proxy model vẫn dùng BCE thuần (không cần XAI constraint
trên model được aggregate).


In [ ]:
def train_one_local_epoch_mutual_xai(
    client: MNASClientState,
    cfg: MNASRuntimeConfig,
    device: torch.device,
    xai_criterion: SaliencyLearningLoss,
    expert_mask: torch.Tensor,
) -> Dict[str, float]:
    """Mutual learning + XAI saliency penalty trên personalized model."""
    bce_criterion = nn.BCEWithLogitsLoss()
    client.personalized_model.train()
    client.proxy_model.train()

    total_personal = 0.0
    total_proxy    = 0.0
    total_base     = 0.0
    total_sal_pen  = 0.0
    total_n = 0

    mask = expert_mask.to(device)

    for xb, yb in client.train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        client.opt_personalized.zero_grad(set_to_none=True)
        client.opt_proxy.zero_grad(set_to_none=True)

        # Personalized model: XAI loss (BCE + saliency penalty)
        loss_p, base_l, sal_pen = xai_criterion(
            client.personalized_model, xb, yb, mask
        )

        # Proxy model: standard BCE
        logits_r = client.proxy_model(xb)
        loss_r = bce_criterion(logits_r, yb)

        # Mutual distillation
        with torch.no_grad():
            logits_p_sg = client.personalized_model(xb)
        dist_w = _adaptive_distillation_loss(
            logits_personal=logits_p_sg,
            logits_proxy=logits_r,
            label_loss_personal=loss_p,
            label_loss_proxy=loss_r,
            temperature=cfg.distill_temperature,
        )

        total_loss_p = loss_p + cfg.distill_weight * dist_w
        total_loss_r = loss_r + cfg.distill_weight * dist_w

        total_loss_p.backward(retain_graph=False)
        total_loss_r.backward()

        client.opt_personalized.step()
        client.opt_proxy.step()

        bs = xb.size(0)
        total_personal += float(total_loss_p.item()) * bs
        total_proxy    += float(total_loss_r.item()) * bs
        total_base     += float(base_l.item()) * bs
        total_sal_pen  += float(sal_pen.item()) * bs
        total_n += bs

    denom = max(total_n, 1)
    return {
        "loss_personal":    total_personal / denom,
        "loss_proxy":       total_proxy    / denom,
        "base_loss":        total_base     / denom,
        "saliency_penalty": total_sal_pen  / denom,
    }


print("train_one_local_epoch_mutual_xai defined")


## XAI — Full FL Pipeline với Saliency Penalty


In [ ]:
def run_mnas_experiment_xai_for_n_clients(
    num_clients: int,
    x_all: np.ndarray,
    y_all: np.ndarray,
    primary_labels_local: np.ndarray,
    label_names_local,
    cfg: MNASRuntimeConfig,
    xai_criterion: SaliencyLearningLoss,
    expert_mask: torch.Tensor,
    show_progress: bool = True,
    client_progress_interval: int = 5,
) -> Dict[str, Any]:
    """MNAS FL experiment với XAI saliency penalty trên personalized model."""
    t_start = time.time()
    if show_progress:
        print_section(f"MNAS+XAI | {num_clients} clients")

    part_pack = build_partition_with_retry(
        labels_for_partition=primary_labels_local,
        y_multi=y_all, label_names=label_names_local,
        num_clients=num_clients, cfg=cfg,
    )
    partition = part_pack["partition"]
    stat_df   = part_pack["stats"]

    if show_progress:
        print_client_split_summary(num_clients, stat_df, {
            "best_seed": part_pack["seed"],
            "score": part_pack["score"],
        })

    dataset           = FullTabularDataset(x_all, y_all)
    client_loaders    = build_client_dataloaders(dataset, partition, cfg.batch_size, cfg.num_workers, cfg.pin_memory)
    global_eval_loader = build_global_eval_dataloader(dataset, cfg.batch_size, cfg.num_workers, cfg.pin_memory)

    input_dim  = x_all.shape[1]
    num_labels = y_all.shape[1]
    sanity_check_shapes(x_all, y_all, num_labels=num_labels, feature_dim=input_dim)

    op_cost = _build_operation_cost_table()
    device  = torch.device(cfg.device)

    if show_progress:
        print_subsection("Architecture Search (MNAS+XAI)")

    client_states: Dict[int, MNASClientState] = {}
    for cpos, (cid, loader) in enumerate(client_loaders.items(), 1):
        selected_ops = search_personalized_architecture(loader, input_dim, num_labels, cfg, op_cost, device)
        personalized = PersonalizedSearchedLCSMC(input_dim, num_labels, cfg.hidden_channels, cfg.attention_reduction, selected_ops).to(device)
        proxy        = ProxyLCSMC(input_dim, num_labels, cfg.hidden_channels, cfg.attention_reduction).to(device)
        client_states[cid] = MNASClientState(
            client_id=cid, train_loader=loader,
            num_samples=int(len(partition[cid])), selected_ops=selected_ops,
            personalized_model=personalized, proxy_model=proxy,
            opt_personalized=_build_optimizer(personalized, cfg),
            opt_proxy=_build_optimizer(proxy, cfg),
            label_coverage=get_label_coverage_for_client(partition[cid], y_all),
        )
        if show_progress and (cpos % max(1, client_progress_interval) == 0 or cpos == len(client_loaders)):
            print(f"search_progress={cpos}/{len(client_loaders)} | client_id={cid}")

    global_proxy_state = copy.deepcopy(client_states[sorted(client_states.keys())[0]].proxy_model.state_dict())
    round_rows: List[Dict[str, Any]] = []
    selected_client_ids = sorted(client_states.keys())

    for rnd in range(1, cfg.global_rounds + 1):
        active_ids = selected_client_ids
        if cfg.client_fraction < 1.0:
            m = max(1, int(len(selected_client_ids) * cfg.client_fraction))
            active_ids = sorted(np.random.choice(selected_client_ids, size=m, replace=False).tolist())

        if show_progress:
            print_round_start(rnd, cfg.global_rounds, len(active_ids))

        for cid in active_ids:
            client_states[cid].proxy_model.load_state_dict(global_proxy_state, strict=True)

        uploaded_states = []
        personal_losses = []
        proxy_losses    = []
        sal_penalties   = []

        for pos, cid in enumerate(active_ids, 1):
            st  = client_states[cid]
            out = {}
            for _ in range(cfg.local_epochs):
                out = train_one_local_epoch_mutual_xai(
                    st, cfg=cfg, device=device,
                    xai_criterion=xai_criterion, expert_mask=expert_mask,
                )
            personal_losses.append(out["loss_personal"])
            proxy_losses.append(out["loss_proxy"])
            sal_penalties.append(out["saliency_penalty"])
            uploaded_states.append((copy.deepcopy(st.proxy_model.state_dict()), st.num_samples))

            if show_progress and (pos % max(1, client_progress_interval) == 0 or pos == len(active_ids)):
                print_client_training_progress(rnd, pos, len(active_ids), cid, "xai_mutual",
                                               out["loss_personal"], out["loss_proxy"])

        global_proxy_state = fedavg_proxy_states(uploaded_states)

        eval_proxy = ProxyLCSMC(input_dim, num_labels, cfg.hidden_channels, cfg.attention_reduction).to(device)
        eval_proxy.load_state_dict(global_proxy_state, strict=True)
        metrics = evaluate_global_model(eval_proxy, global_eval_loader, cfg.multilabel_threshold, device)

        round_rows.append({
            "num_clients": num_clients, "round": rnd, **metrics,
            "mean_loss_personal":    float(np.mean(personal_losses)),
            "mean_loss_proxy":       float(np.mean(proxy_losses)),
            "mean_saliency_penalty": float(np.mean(sal_penalties)),
            "active_clients": len(active_ids),
        })

        if show_progress:
            print_round_metrics(rnd, metrics)
            print(f"  saliency_penalty_avg={round_rows[-1]['mean_saliency_penalty']:.6f}")

    if show_progress:
        print_subsection("Final Personalized Fine-tuning (XAI)")
    for cid in sorted(client_states.keys()):
        fine_tune_personalized_model(client_states[cid], cfg=cfg, device=device)

    history_df    = pd.DataFrame(round_rows)
    final_metrics = history_df.iloc[-1][["loss","precision","recall","f1","precision_micro","recall_micro","f1_micro"]].to_dict()
    elapsed       = time.time() - t_start

    if show_progress:
        print_experiment_summary(num_clients=num_clients, final_metrics=final_metrics, elapsed_sec=elapsed)

    mdir = OUT / "outputs" / "metrics"
    cdir = OUT / "outputs" / "checkpoints"
    mdir.mkdir(parents=True, exist_ok=True)
    cdir.mkdir(parents=True, exist_ok=True)

    history_df.to_csv(mdir / f"mnas_xai_round_metrics_{num_clients}clients.csv", index=False)
    history_df.to_json(mdir / f"mnas_xai_round_metrics_{num_clients}clients.json", orient="records", indent=2)
    torch.save({"global_proxy_state": global_proxy_state, "num_clients": num_clients},
               cdir / f"mnas_xai_global_proxy_{num_clients}clients.pt")

    return {
        "num_clients": num_clients, "history_df": history_df,
        "split_stats": stat_df, "final_metrics": final_metrics,
        "elapsed_sec": elapsed, "client_states": client_states,
    }


def run_all_mnas_xai_scenarios(x_all, y_all, primary_labels_local, label_names_local,
                                cfg, xai_criterion, expert_mask,
                                show_progress=True, client_progress_interval=5):
    results = {}
    for n in cfg.scenarios:
        results[int(n)] = run_mnas_experiment_xai_for_n_clients(
            num_clients=int(n), x_all=x_all, y_all=y_all,
            primary_labels_local=primary_labels_local,
            label_names_local=label_names_local, cfg=cfg,
            xai_criterion=xai_criterion, expert_mask=expert_mask,
            show_progress=show_progress, client_progress_interval=client_progress_interval,
        )
    return results


print("XAI pipeline: run_mnas_experiment_xai_for_n_clients, run_all_mnas_xai_scenarios defined")


## XAI — Run & Visualize Saliency


In [ ]:
RUN_XAI_TRAINING = True   # Set False để bỏ qua training XAI

if RUN_XAI_TRAINING:
    xai_results = run_all_mnas_xai_scenarios(
        x_all=X_all, y_all=Y_all,
        primary_labels_local=primary_labels,
        label_names_local=label_names,
        cfg=CFG,
        xai_criterion=XAI_CRITERION,
        expert_mask=EXPERT_MASK,
        show_progress=True,
        client_progress_interval=5,
    )
    print("XAI scenarios done:", list(xai_results.keys()))
else:
    print("RUN_XAI_TRAINING=False — skipping XAI training.")
    xai_results = {}


In [ ]:
# ─── Visualize Saliency (chạy sau khi XAI training xong) ─────────────────────
device_viz = torch.device(CFG.device)

for n_clients, pack in xai_results.items():
    ckpt_file = OUT / "outputs" / "checkpoints" / f"mnas_xai_global_proxy_{n_clients}clients.pt"
    ckpt = torch.load(str(ckpt_file), map_location=device_viz)

    viz_model = ProxyLCSMC(
        input_dim=X_all.shape[1], num_labels=Y_all.shape[1],
        channels=CFG.hidden_channels, reduction=CFG.attention_reduction,
    ).to(device_viz)
    viz_model.load_state_dict(ckpt["global_proxy_state"], strict=True)

    # Dùng 50k samples đầu để tránh OOM khi visualize
    viz_dataset = FullTabularDataset(X_all[:50_000], Y_all[:50_000])
    viz_loader  = build_global_eval_dataloader(viz_dataset, batch_size=512, num_workers=0, pin_memory=False)

    viz = SaliencyVisualizer(viz_model, feature_columns, device_viz)
    stats = viz.aggregate(viz_loader, max_batches=50)

    fig_path = str(OUT / "outputs" / "figures" / f"xai_saliency_{n_clients}clients.png")
    viz.plot_bar(
        stats,
        title=f"XAI Feature Saliency — {n_clients} clients  (red=penalized by expert mask)",
        expert_mask=EXPERT_MASK,
        save_path=fig_path,
    )
    print(f"Saved saliency chart → {fig_path}")
